# 02 — Exploratory analysis of hourly PM2.5


Exploration is conducted on the cleaned series while missing intervals remain visible. The goal is to characterize distribution, periodicity, persistence, regime changes and high-concentration episodes—not to tune against the locked test set.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
RAW=ROOT/"data/raw/London_Marylebone_PM25.csv"
PROCESSED=ROOT/"data/processed"
FIGURES=ROOT/"figures"
RESULTS=ROOT/"results"
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pm25forecast.pipeline import prepare
from pm25forecast.viz import data_quality_figures

In [ ]:
if not (PROCESSED/"pm25_model_ready.csv").exists(): prepare(RAW,ROOT)
df=pd.read_csv(PROCESSED/"pm25_model_ready.csv",parse_dates=["datetime"])
df[["pm25_clean","gap_length_hours","instrument_type"]].describe(percentiles=[.5,.75,.9,.95,.99])

In [ ]:
daily=df.set_index("datetime").pm25_clean.resample("D").mean()
ax=daily.plot(figsize=(15,4),lw=.7,title="Daily mean PM2.5"); ax.set_ylabel("µg/m³");

In [ ]:
profiles=pd.concat({
 "hour":df.groupby(df.datetime.dt.hour).pm25_clean.mean(),
 "weekday":df.groupby(df.datetime.dt.dayofweek).pm25_clean.mean(),
 "month":df.groupby(df.datetime.dt.month).pm25_clean.mean()},axis=1)
profiles

In [ ]:
lags=[1,2,3,6,12,24,48,72,168,336,720]
pd.Series({lag:df.pm25_clean.autocorr(lag) for lag in lags},name="autocorrelation").to_frame()

In [ ]:
df.groupby("instrument_name").pm25_clean.agg(["count","mean","median","std",lambda s:s.quantile(.9)])

Instrument differences are descriptive, not causal: the transition is confounded with calendar time, emissions and policy changes. Instrument type is retained as a regime indicator rather than used to “correct” historical readings.

In [ ]:
data_quality_figures(df,FIGURES)
sorted(p.name for p in FIGURES.glob("0*.png"))